In [1]:
# NB14_attr43_profile.ipynb

# ---------------------------------------------------------------------------
# NB14 - Attr43 Profile and Attr43 x System-Grade Bad-Rate Cross-Matrix
#
# Purpose:
#   Reviewer comment (Section 3.3): under data-generating mechanism DGM-b1 the
#   upgrade probability is tied to Attr43. The reviewer asks (i) for a
#   description of Attr43 and (ii) for a cross-matrix of Attr43 against the
#   system credit grade based on the bad (default) rate, to visualise the
#   relationship between the two.
#
# Attr43 in the Polish Companies Bankruptcy dataset:
#   Attr43 is an activity / operating-cycle ratio ("rotation receivables +
#   inventory turnover, in days"). It is NOT a leverage or profitability
#   signal, which is precisely why it was selected as the DGM-b1 driver: it
#   is near-orthogonal to the system default-probability estimate (pd_system)
#   and to realised default. This notebook documents that near-orthogonality
#   empirically (Spearman correlations) and shows the bad-rate cross-matrix.
#
# Input:   ../data/processed/graded_data.parquet
# Output:  ../results/tables/21_attr43_profile.csv
#          ../results/tables/21b_attr43_grade_badrate_matrix.csv
#          ../results/figures/32_attr43_grade_badrate.{png,pdf}
#          ../results/figures/32b_attr43_marginal_badrate.{png,pdf}
# ---------------------------------------------------------------------------

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 600, "font.family": "serif",
    "axes.edgecolor": "black", "axes.linewidth": 0.8,
    "grid.color": "0.85", "grid.linewidth": 0.5,
})
GREY_DARK, GREY_MID, GREY_LIGHT = "0.20", "0.45", "0.70"

FIG_DIR, TABLE_DIR = "../results/figures/", "../results/tables/"
os.makedirs(FIG_DIR, exist_ok=True); os.makedirs(TABLE_DIR, exist_ok=True)

def savefig(fig, name):
    fig.savefig(FIG_DIR + name + ".png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG_DIR + name + ".pdf", bbox_inches="tight")
    plt.close(fig)

# ---- 1. Load data ---------------------------------------------------------
g = pd.read_parquet("../data/processed/graded_data.parquet")
GRADE_LABELS = ["AAA", "AA", "A", "BBB", "BB", "B", "CCC"]
a43 = g["Attr43"].astype(float)
y   = g["default"].astype(int)
print(f"N = {len(g):,}  Attr43 missing = {a43.isna().sum():,}")

# ---- 2. Attr43 distribution profile + near-orthogonality evidence ---------
desc = a43.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
rho_pd, p_pd = spearmanr(a43, g["pd_system"], nan_policy="omit")
rho_def, p_def = spearmanr(a43, y, nan_policy="omit")
profile = pd.DataFrame({
    "statistic": ["count", "mean", "std", "min", "p05", "p25", "median",
                  "p75", "p95", "max",
                  "spearman_rho_vs_pd_system", "spearman_p_vs_pd_system",
                  "spearman_rho_vs_default", "spearman_p_vs_default"],
    "value": [desc["count"], desc["mean"], desc["std"], desc["min"],
              desc["5%"], desc["25%"], desc["50%"], desc["75%"], desc["95%"],
              desc["max"], rho_pd, p_pd, rho_def, p_def],
})
profile["value"] = profile["value"].astype(float).round(6)
profile.to_csv(TABLE_DIR + "21_attr43_profile.csv", index=False)
print("\nAttr43 profile (note near-zero Spearman rho vs pd_system and default):")
print(profile.to_string(index=False))

# ---- 3. Attr43 quintile x system grade -> bad-rate cross-matrix -----------
q5 = pd.qcut(a43.rank(method="first"), 5,
             labels=["Q1 (low)", "Q2", "Q3", "Q4", "Q5 (high)"])
df = pd.DataFrame({"attr43_q": q5,
                   "grade": pd.Categorical(g["system_grade"], categories=GRADE_LABELS,
                                           ordered=True),
                   "default": y})
badrate = (df.pivot_table(index="attr43_q", columns="grade",
                          values="default", aggfunc="mean", observed=False) * 100)
counts  = df.pivot_table(index="attr43_q", columns="grade",
                         values="default", aggfunc="size", observed=False)
badrate = badrate.reindex(columns=GRADE_LABELS)
badrate.round(3).to_csv(TABLE_DIR + "21b_attr43_grade_badrate_matrix.csv")
print("\nBad-rate (%) cross-matrix  (rows: Attr43 quintile, cols: system grade):")
print(badrate.round(2).to_string())

# ---- 4. Figure 32: bad-rate heatmap (Greys) -------------------------------
# Reading left->right (grade worsens) the bad rate rises; reading top->bottom
# (Attr43 rises) it is essentially flat: Attr43 carries no default signal.
fig, ax = plt.subplots(figsize=(7.2, 3.8))
sns.heatmap(badrate, cmap="YlOrRd", annot=True, fmt=".1f",
            annot_kws={"size": 8}, linewidths=0.5, linecolor="white",
            cbar_kws={"label": "Default rate (%)"}, ax=ax)
ax.set_xlabel("System rating grade")
ax.set_ylabel("Attr43 quintile")
savefig(fig, "32_attr43_grade_badrate")

# ---- 5. Figure 32b: marginal default rate by Attr43 quintile (flat) -------
marg = df.groupby("attr43_q", observed=False)["default"].mean() * 100
fig, ax = plt.subplots(figsize=(6.0, 3.8))
ax.bar(range(len(marg)), marg.values, color=GREY_MID, edgecolor="black",
       linewidth=0.7)
ax.axhline(y.mean() * 100, color="black", linestyle="--", linewidth=1.0)
ax.set_xticks(range(len(marg)))
ax.set_xticklabels(marg.index, rotation=0)
ax.set_xlabel("Attr43 quintile")
ax.set_ylabel("Default rate (%)")
for i, v in enumerate(marg.values):
    ax.text(i, v + 0.05, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
savefig(fig, "32b_attr43_marginal_badrate")

print("\nNB14 complete: Attr43 profile, bad-rate matrix, two figures saved.")


N = 43,405  Attr43 missing = 0

Attr43 profile (note near-zero Spearman rho vs pd_system and default):
                statistic        value
                    count 43405.000000
                     mean   117.847780
                      std    91.020987
                      min     9.690959
                      p05    28.633400
                      p25    66.702000
                   median    99.401500
                      p75   140.540000
                      p95   259.702000
                      max   659.646100
spearman_rho_vs_pd_system    -0.026743
  spearman_p_vs_pd_system     0.000000
  spearman_rho_vs_default     0.007554
    spearman_p_vs_default     0.115537

Bad-rate (%) cross-matrix  (rows: Attr43 quintile, cols: system grade):
grade       AAA    AA     A   BBB    BB      B    CCC
attr43_q                                             
Q1 (low)   0.52  0.53  0.77  2.21  4.04   9.30  18.27
Q2         0.58  0.78  0.72  2.27  3.78   8.00  16.87
Q3         0.12  1.37  


NB14 complete: Attr43 profile, bad-rate matrix, two figures saved.
